# 🛒 Store Sales Time Series Forecasting
### LightGBM ile Geliştirilmiş Satış Tahmin Modeli

**İyileştirmeler:**
- Tatil/etkinlik verisi eklendi (`holidays_events.csv`)
- İşlem hacmi (`transactions.csv`) özellik olarak eklendi
- Daha fazla lag ve rolling window özelliği
- Kategorik encoding iyileştirildi (LightGBM native)
- Özellik önem analizi eklendi
- RMSLE doğrudan optimize edildi (tweedie/poisson yerine log-transform + RMSE)
- Seed sabitlendi → tekrarlanabilir sonuçlar
- Bellek optimizasyonu (dtype downcasting)

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import gc
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error

SEED = 42
np.random.seed(SEED)

BASE_PATH = '/kaggle/input/competitions/store-sales-time-series-forecasting/'

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# ─── 1. VERİ YÜKLEME ──────────────────────────────────────────────
train       = pd.read_csv(f'{BASE_PATH}train.csv',           parse_dates=['date'])
test        = pd.read_csv(f'{BASE_PATH}test.csv',            parse_dates=['date'])
oil         = pd.read_csv(f'{BASE_PATH}oil.csv',             parse_dates=['date'])
stores      = pd.read_csv(f'{BASE_PATH}stores.csv')
holidays    = pd.read_csv(f'{BASE_PATH}holidays_events.csv', parse_dates=['date'])
transactions= pd.read_csv(f'{BASE_PATH}transactions.csv',    parse_dates=['date'])

print("Train shape:",  train.shape)
print("Test shape:",   test.shape)
print("Train tarih aralığı:", train.date.min(), "→", train.date.max())
print("Test  tarih aralığı:", test.date.min(),  "→", test.date.max())

In [ ]:
# ─── 2. ÖN İŞLEME ─────────────────────────────────────────────────

# Petrol fiyatları: ileri + geri doldur, ardından 7 günlük hareketli ortalama ekle
oil = oil.rename(columns={'dcoilwtico': 'oil_price'})
oil['oil_price'] = oil['oil_price'].ffill().bfill()
oil['oil_ma7']   = oil['oil_price'].rolling(7, min_periods=1).mean()

# Tatiller: sadece ulusal tatilleri al, transfer/köprü günü olup olmadığını işaretle
nat_hol = holidays[
    (holidays['locale'] == 'National') & 
    (holidays['transferred'] == False)
][['date']].drop_duplicates()
nat_hol['is_holiday'] = 1

# Transactions: mağaza başına günlük ortalama işlem
transactions = transactions.rename(columns={'transactions': 'txn'})

print("Ön işleme tamamlandı.")

In [ ]:
# ─── 3. ÖZELLİK MÜHENDİSLİĞİ ────────────────────────────────────

def reduce_mem(df):
    """Bellek kullanımını azalt (int64→int32, float64→float32)"""
    for col in df.select_dtypes('int64').columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    return df

def prepare_data(train_df, test_df, stores_df, oil_df, holidays_df, txn_df):
    full_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # Birleştirmeler
    full_df = full_df.merge(stores_df,   on='store_nbr', how='left')
    full_df = full_df.merge(oil_df,      on='date',      how='left')
    full_df = full_df.merge(holidays_df, on='date',      how='left')
    full_df = full_df.merge(txn_df,      on=['date', 'store_nbr'], how='left')

    full_df['oil_price'] = full_df['oil_price'].ffill()
    full_df['oil_ma7']   = full_df['oil_ma7'].ffill()
    full_df['is_holiday'] = full_df['is_holiday'].fillna(0).astype('int8')

    # Zaman özellikleri
    full_df['month']       = full_df['date'].dt.month.astype('int8')
    full_df['day_of_week'] = full_df['date'].dt.dayofweek.astype('int8')
    full_df['day_of_month']= full_df['date'].dt.day.astype('int8')
    full_df['year']        = full_df['date'].dt.year.astype('int16')
    full_df['week_of_year']= full_df['date'].dt.isocalendar().week.astype('int8')
    full_df['quarter']     = full_df['date'].dt.quarter.astype('int8')
    full_df['is_weekend']  = (full_df['day_of_week'] >= 5).astype('int8')
    full_df['is_payday']   = (
        (full_df['date'].dt.day == 15) | (full_df['date'].dt.is_month_end)
    ).astype('int8')
    full_df['is_month_start'] = full_df['date'].dt.is_month_start.astype('int8')

    # Kategorik encode (LightGBM native categorical için category dtype)
    le = LabelEncoder()
    for col in ['family', 'city', 'state', 'type']:
        full_df[col] = le.fit_transform(full_df[col].astype(str))

    # Log-transform satışlar
    full_df['sales'] = np.log1p(full_df['sales'].clip(lower=0))

    # ── LAG ÖZELLİKLERİ ──
    grp = full_df.groupby(['store_nbr', 'family'])['sales']
    for lag in [16, 21, 28, 35, 42]:
        full_df[f'lag_{lag}'] = grp.transform(lambda x: x.shift(lag))

    # ── ROLLING WINDOW ──
    for window in [7, 14, 28]:
        full_df[f'roll_mean_{window}'] = grp.transform(
            lambda x: x.shift(16).rolling(window, min_periods=1).mean()
        )
        full_df[f'roll_std_{window}']  = grp.transform(
            lambda x: x.shift(16).rolling(window, min_periods=1).std().fillna(0)
        )

    # ── EXPONENTIAL WEIGHTED MEAN ──
    full_df['ewm_alpha04'] = grp.transform(
        lambda x: x.shift(16).ewm(alpha=0.4, adjust=False).mean()
    )

    # Transactions lag
    txn_grp = full_df.groupby('store_nbr')['txn']
    full_df['txn_lag16'] = txn_grp.transform(lambda x: x.shift(16))
    full_df['txn_roll7'] = txn_grp.transform(
        lambda x: x.shift(16).rolling(7, min_periods=1).mean()
    )

    full_df = reduce_mem(full_df)
    return full_df

print("Özellik mühendisliği hazırlanıyor...")
full_data = prepare_data(train, test, stores, oil, nat_hol, transactions)
print("Tamamlandı! Shape:", full_data.shape)

# ── ÖZELLİK LİSTESİ ──
features = [
    'store_nbr', 'family', 'onpromotion',
    'month', 'day_of_week', 'day_of_month', 'year',
    'week_of_year', 'quarter', 'is_weekend', 'is_payday', 'is_month_start',
    'is_holiday', 'oil_price', 'oil_ma7', 'cluster',
    # Lag
    'lag_16', 'lag_21', 'lag_28', 'lag_35', 'lag_42',
    # Rolling
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28',
    'roll_std_7', 'roll_std_14', 'roll_std_28',
    'ewm_alpha04',
    # Transactions
    'txn_lag16', 'txn_roll7'
]

# Train / Test ayır
train_final = full_data[(full_data['sales'].notnull()) & (full_data['date'] >= '2013-02-01')].copy()
test_final  = full_data[full_data['sales'].isnull()].copy()

print("Train final:", train_final.shape)
print("Test  final:", test_final.shape)
gc.collect()

In [ ]:
# ─── 4. MODEL EĞİTİMİ (Time-Based Split) ─────────────────────────
SPLIT_DATE = '2017-07-01'  # Daha fazla validation verisi

X_train = train_final[train_final.date < SPLIT_DATE][features].values
y_train = train_final[train_final.date < SPLIT_DATE]['sales'].values

X_val   = train_final[train_final.date >= SPLIT_DATE][features].values
y_val   = train_final[train_final.date >= SPLIT_DATE]['sales'].values

feature_names = features  # lgb için

dtrain = lgb.Dataset(X_train, label=y_train, feature_name=features)
dval   = lgb.Dataset(X_val,   label=y_val,   reference=dtrain, feature_name=features)

params = {
    'objective':        'regression_l1',    # MAE → outlier'lara dayanıklı
    'metric':           'rmse',
    'verbosity':        -1,
    'boosting_type':    'gbdt',
    'learning_rate':    0.05,
    'num_leaves':       127,
    'max_depth':        -1,
    'min_child_samples':20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'lambda_l1':        0.1,
    'lambda_l2':        0.1,
    'seed':             SEED,
    'n_jobs':           -1,
}

model = lgb.train(
    params,
    dtrain,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'valid'],
    num_boost_round=5000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=150, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

In [ ]:
# ─── 5. DEĞERLENDİRME ─────────────────────────────────────────────
val_preds = model.predict(X_val)

# Negatif tahminleri 0'a klample
val_preds_clipped = np.clip(val_preds, 0, None)

val_score = np.sqrt(mean_squared_log_error(
    np.expm1(y_val),
    np.expm1(val_preds_clipped)
))
print(f"✅ Validation RMSLE: {val_score:.5f}")

# ─── Özellik Önem Grafiği ───
fi = pd.Series(model.feature_importance('gain'), index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
fi.head(20).plot.barh(ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 20 Feature Importance (Gain)', fontsize=14)
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print("\nTop 10 özellik:")
print(fi.head(10).to_string())

In [ ]:
# ─── 6. TAHMİN VE SUBMISSION ──────────────────────────────────────
test_preds = model.predict(test_final[features].values)
test_preds = np.clip(test_preds, 0, None)  # negatif değerleri sıfırla

submission = pd.DataFrame({
    'id':    test_final['id'].astype(int),
    'sales': np.expm1(test_preds)
})

submission.to_csv('submission.csv', index=False)
print("✅ Submission başarıyla kaydedildi! Satır sayısı:", len(submission))
submission.head()

In [ ]:
# ─── 7. MODELİ KAYDET ─────────────────────────────────────────────
model.save_model('lgb_store_sales.txt')
print("✅ Model 'lgb_store_sales.txt' olarak kaydedildi.")
print(f"   Best iteration: {model.best_iteration}")
print(f"   Özellik sayısı: {len(features)}")